# Understanding the data and experimenting with collection of external data

In [26]:
import pandas as pd
import numpy as np
import seaborn as sns

## Import and view data

In [4]:
cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

In [7]:
cross_border_payments.head()

,transaction_id,entity_id,entity_name,sector,date,direction,currency_pair,value_zar,counterparty_country,corridor_type,beneficiary_name,reference,memo
0,XBP63220455,E01,BHP Group,mining,2023-07-01,inbound,USD/ZAR,2541553.55,Switzerland,intercompany,BHP Group Switzerland Ltd,INTERCO-730855,NaN
1,XBP14207725,E11,Pepkor Holdings,consumer,2023-07-01,outbound,USD/ZAR,407839.87,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-827118,NaN
2,XBP66460952,E11,Pepkor Holdings,consumer,2023-07-01,outbound,CNY/ZAR,72148.47,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-488519,NaN
3,XBP45973312,E11,Pepkor Holdings,consumer,2023-07-01,outbound,GBP/ZAR,53285.65,Namibia,intercompany,Pepkor Holdings Namibia Ltd,INTERCO-585129,NaN
4,XBP13173829,E11,Pepkor Holdings,consumer,2023-07-01,inbound,AED/ZAR,2858193.91,Japan,trade,Continental Resources Trading,TRADE-568825,NaN


In [8]:
trade_finance.head()

,instrument_id,entity_id,entity_name,sector,date,instrument_type,direction,tenor_days,value_zar,counterparty_country,commodity_or_contract_type,status,beneficiary_name,reference,memo
0,TF67401938,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,60,13499920.55,United Arab Emirates,agri_produce,issued,Silverline Trading Co.,LC-471415,NaN
1,TF91455580,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,365,558304.97,Switzerland,iron_ore,settled,Global Commodities Marketing,LC-266865,NaN
2,TF31370953,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,30,4042335.97,United Kingdom,agri_produce,settled,Pacific International Trading House,LC-946978,NaN
3,TF13634137,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,import,365,171042.89,China,platinum_group_metals,active,Silverline Resources Trading,LC-553503,NaN
4,TF86438695,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,120,331531.78,Netherlands,electronics,settled,Silverline Resources Trading,LC-260628,NaN


In [9]:
transactional_banking.head()

,transaction_id,entity_id,entity_name,sector,date,leg_type,direction,amount_zar,currency,channel,beneficiary_name,reference,memo
0,TXN40610803,E01,BHP Group,mining,2023-07-01,collections,inbound,63878.473869,ZAR,EFT,Continental Metals Trading House,INV-662227,NaN
1,TXN12547643,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,53220.970000,ZAR,EFT,Sunrise Cold Chain Logistics,INV-591371,NaN
2,TXN37710224,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,12309.970000,ZAR,SWIFT,Sunrise Cold Chain Logistics,PO-687736,NaN
3,TXN65618575,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,6308.300000,ZAR,Internal Transfer,Sunrise Cold Chain Logistics,INV-139304,NaN
4,TXN50222056,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,55346.750000,ZAR,Internal Transfer,Cape Wholesale Distributors,INV-556985,NaN


In [32]:
list(set(transactional_banking.entity_name.unique()) | \
    set(cross_border_payments.entity_name.unique()) | \
    set(trade_finance.entity_name.unique()))

['Sanlam',
 'Vodacom Group',
 'OUTsurance Group',
 'Valterra Platinum',
 'Naspers',
 'Shaftesbury Capital plc',
 'NEPI Rockcastle',
 'Gold Fields',
 'BHP Group',
 'Glencore',
 'AngloGold Ashanti',
 'Shoprite Holdings',
 'Clicks Group',
 'Prosus',
 'Bid Corporation',
 'Pepkor Holdings',
 'Anglo American',
 'Aspen Pharmacare',
 'The Bidvest Group',
 'MTN Group']

Interesting, we only have 20 companies...

#TODO: Double check
My understanding of the data is as follows:
- Transactional: Actual payments made by companies
- Cross border payments: Message sent between banks, used when money is transfered internationally 
(e.g. Syn bank sends a swift message to Barclays that Pepkor transfered 1000 pounds to jane street)
- Trade Finance: Basically communication to ensure that buyers and sellers are protected. It'll 
store things like company a is buying product from company b, then a credit note is reached out to 
tell company b that it will get payed once the product is shipped.

**This means that the same transaction can appear in all 3 datasets, so we need to figure out how 
to remove the same transaction**

**We need to find out what the core product pillars are for wich we are predicting wallet share**

## Retrieving external data

We need to get external data to actually predict wallet size

The goal of this section is not to retrieve data, but more to experiment on possible ways to get 
the data. Once methods are finalized, a Python script will be written to retrieve external data.

In [4]:
# Tickers string for companies, used with yahoo finance
tickers_str = "SLM.JO VOD.JO OUT.JO VAL.JO NPN.JO SHC.JO NRP.JO GFI.JO BHG.JO GLN.JO ANG.JO SHP.JO " \
"CLS.JO PRX.JO BVT.JO PPH.JO AGL.JO APN.JO BTI.JO MTN.JO"

#Used for getting ticker data
company_tickers = {
    "Sanlam": "SLM.JO",
    "Vodacom Group": "VOD.JO",
    "OUTsurance Group": "OUT.JO",
    "Valterra Platinum": "VAL.JO",
    "Naspers": "NPN.JO",
    "Shaftesbury Capital plc": "SHC.JO",
    "NEPI Rockcastle": "NRP.JO",
    "Gold Fields": "GFI.JO",
    "BHP Group": "BHG.JO",
    "Glencore": "GLN.JO",
    "AngloGold Ashanti": "ANG.JO",
    "Shoprite Holdings": "SHP.JO",
    "Clicks Group": "CLS.JO",
    "Prosus": "PRX.JO",
    "Bid Corporation": "BVT.JO",
    "Pepkor Holdings": "PPH.JO",
    "Anglo American": "AGL.JO",
    "Aspen Pharmacare": "APN.JO",
    "The Bidvest Group": "BTI.JO",
    "MTN Group": "MTN.JO",
}

In [2]:
import yfinance as yf

portfolio = yf.Tickers(tickers_str)

In [9]:
dir(portfolio.tickers[company_tickers["Sanlam"]])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_analysis',
 '_data',
 '_download_options',
 '_earnings',
 '_earnings_dates',
 '_expirations',
 '_fast_info',
 '_fetch_ticker_tz',
 '_financials',
 '_fundamentals',
 '_funds_data',
 '_get_earnings_dates_using_scrape',
 '_get_earnings_dates_using_screener',
 '_get_ticker_tz',
 '_holders',
 '_isin',
 '_lazy_load_price_history',
 '_message_handler',
 '_news',
 '_options2df',
 '_price_history',
 '_quote',
 '_shares',
 '_tz',
 '_underlying',
 'actions',
 'analyst_price_targets',
 'balance_sheet',
 'balancesheet',
 'calendar',
 'capital_gains',
 'cash_flow',
 'cashflow',
 'dividends',
 'earnings',
 'earnings_d

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.scripts.data_processing import Data_Processor

In [3]:
my_data_processor = Data_Processor()

In [ ]:
#company_res, sens_res = my_data_processor.extract_external_data_from_pdfs('../data/downloads/Pepkor_Holdings/')

10:51:40 | INFO | Pdf paths: [PosixPath('../data/downloads/Pepkor_Holdings/annual_report__Pepkor-integrated-report-2025.pdf'), PosixPath('../data/downloads/Pepkor_Holdings/financial_statements__Pepkor-annual-financial-statements-2025.pdf'), PosixPath('../data/downloads/Pepkor_Holdings/interim_results__Pepkor-interim-results-for-the-six-months-ended-31-March-2025.pdf'), PosixPath('../data/downloads/Pepkor_Holdings/results_presentation__Short-form-for-the-six-months-ended-31-March-2025.pdf')]
10:52:18 | INFO | Pdf paths: [PosixPath('../data/downloads/Pepkor_Holdings/SENS/2026-02-06__S516757__Dealings_in_Securities_by_an_Associate.pdf'), PosixPath('../data/downloads/Pepkor_Holdings/SENS/2026-03-02__S517833__Interest_Payment_Notifications_in_respect_of_Listed_Notes_under_the_DMTN_Programme.pdf'), PosixPath('../data/downloads/Pepkor_Holdings/SENS/2026-01-19__S515880__Notice_of_AGM_and_Availability_of_Integrated_Report_and_Broad-Based_Black_Economic_Empowerment_Report.pdf'), PosixPath('../da

In [12]:
import json

In [ ]:
company_res.model_dump_json()

str

In [32]:
#NOTE: Method now automatically dumps

company_json = json.loads(company_res.model_dump_json())
sens_json = json.loads(sens_res.model_dump_json())

In [35]:
pd.DataFrame(company_json["records"])

,company,report_date,reporting_currency,reporting_unit,source_document,revenue,cost_of_sales,inventory,trade_receivables,trade_payables,...,security_collateral_on_debt,major_contractual_commitments,order_book_project_pipeline,assets_under_management,market_capitalisation,share_price,share_return,enterprise_value,credit_rating,ownership_major_shareholders
0,Pepkor Holdings Limited,2025-09-30,ZAR,Millions,Pepkor Holdings Limited Annual Financial State...,95340.0,57388.0,18618.0,1551.0,12023.0,...,Long- and short-term investments at banking in...,Contracts for capital expenditure amounting to...,None,None,89800.0,2431.0,None,None,None,"[Public Investment Corporation (PIC), Titan Pr..."


In [37]:
pd.DataFrame(sens_json["events"])

,company,announcement_date,title,source_document,source_url,event_type,event_value,currency,counterparty,target_or_asset,country,expected_completion_date,banking_opportunities,opportunity_summary
0,Pepkor Holdings Limited,10 December 2025,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,1.750000e+09,ZAR,Nedbank Limited and Absa Bank Limited,PEP12 and PEP13 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates]",Listing of PEP12 (ZAR 750 million) and PEP13 (...
1,Pepkor Holdings Limited,4 March 2026,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,1.130000e+09,ZAR,Nedbank Limited,PEP14 and PEP15 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates]",Listing of PEP14 (ZAR 595 million) and PEP15 (...
2,Pepkor Holdings Limited,7 March 2025,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,2.083000e+09,ZAR,Rand Merchant Bank,PEP09 and PEP10 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates, credit]",Pepkor raised R2.1 billion via an auction and ...
3,Pepkor Holdings Limited,28 March 2025,LISTING OF NEW FINANCIAL INSTRUMENT,LISTING OF NEW FINANCIAL INSTRUMENT,None,bond_issue,1.250000e+09,ZAR,Absa Bank Limited,PEP11 Senior Unsecured Floating Rate Notes,South Africa,NaN,"[debt_capital_markets, interest_rates, credit]",Placement of R1.25 billion of PEP11 notes to r...
4,Pepkor Holdings Limited,22 July 2026,PEPKOR TO ACQUIRE CONTROLLING STAKE IN TRANSFO...,PEPKOR TO ACQUIRE CONTROLLING STAKE IN TRANSFO...,None,acquisition,1.570000e+09,ZAR,Shop2Shop Proprietary Limited and S2S Africa H...,Shop2Shop Proprietary Limited and Flash Mobile...,South Africa,NaN,"[corporate_finance, credit, fx, payments]",Pepkor to acquire a controlling 57.1% stake in...
5,Pepkor Holdings Limited,25 March 2025,VOLUNTARY ANNOUNCEMENT RELATING TO THE ACQUISI...,VOLUNTARY ANNOUNCEMENT RELATING TO THE ACQUISI...,None,acquisition,NaN,ZAR,Retailability Proprietary Limited,"Legit, Swagga, Style and Boardmans businesses",South Africa,NaN,"[corporate_finance, credit, payments]","Acquisition of Legit, Swagga, Style and Boardm..."
6,Pepkor Holdings Limited,4 November 2025,VOLUNTARY ANNOUNCEMENT RELATING TO THE SUCCESS...,VOLUNTARY ANNOUNCEMENT RELATING TO THE SUCCESS...,None,acquisition,1.700000e+09,ZAR,Retailability Proprietary Limited,"Legit, Swagga, Style and Boardmans businesses",South Africa,2025-11-02,"[corporate_finance, credit, payments]",Successful implementation of the acquisition o...


In [ ]:
data = """{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Anglo_American', 'document_name': 'interim_results__2026__half-year-results-2026-factsheet.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': 'The filename explicitly refers to a factsheet rather than comprehensive interim results.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Anglo_American', 'document_name': 'results_presentation', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent results_presentation document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Aspen_Pharmacare', 'document_name': 'annual_report', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent annual_report document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/BHP_Group', 'document_name': 'results_presentation', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent results_presentation document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Bid_Corporation', 'document_name': 'annual_report__2025__australasia.pdf', 'is_explicitly_incorrect': False, 'possibly_incorrect': True, 'missing_data': False, 'reason': "The filename refers to 'australasia' which is suspicious and may indicate a regional or subsidiary report rather than the group annual report."}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Bid_Corporation', 'document_name': 'financial_statements__2026__sens.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': "The filename explicitly refers to 'sens', indicating an announcement rather than standalone financial statements."}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Bid_Corporation', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Clicks_Group', 'document_name': 'financial_statements__2025__CGL-YE25-Five-year-review.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': 'The filename explicitly refers to a five-year review, which is an unsupported document type.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Glencore', 'document_name': 'annual_report__2026__GLEN-2025-Annual-Report.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': 'The filename contains the year 2025, which explicitly contradicts the expected year 2026 specified in the classification tag.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Glencore', 'document_name': 'financial_statements__2026__GLEN-2025-Annual-Report.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': 'The filename contains the year 2025, which explicitly contradicts the expected year 2026 specified in the classification tag.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Gold_Fields', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/MTN_Group', 'document_name': 'financial_statements__2025__MTN-Holdings-Group-Annual-Financial-Statements-2025-signed.pdf', 'is_explicitly_incorrect': False, 'possibly_incorrect': True, 'missing_data': False, 'reason': "The filename refers to 'MTN-Holdings-Group' which might indicate a subsidiary rather than the listed group entity."}, 
{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/NEPI_Rockcastle', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Naspers', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Pepkor_Holdings', 'document_name': 'results_presentation__2025__Short-form-for-the-six-months-ended-31-March-2025.pdf', 'is_explicitly_incorrect': True, 'possibly_incorrect': False, 'missing_data': False, 'reason': 'The filename explicitly indicates a short-form announcement rather than a presentation.'}, 
{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Prosus', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Prosus', 'document_name': 'results_presentation', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent results_presentation document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Sanlam', 'document_name': 'annual_report__2025__Sanlam-Life-Annual-Report-2025.pdf', 'is_explicitly_incorrect': False, 'possibly_incorrect': True, 'missing_data': False, 'reason': "The filename refers to 'Sanlam-Life', which might be a subsidiary rather than the listed Sanlam Limited group entity."}, 
{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Sanlam', 'document_name': 'results_presentation', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent results_presentation document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Shaftesbury_Capital_plc', 'document_name': 'results_presentation', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent results_presentation document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Shoprite_Holdings', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}, 

{'location': '/home/chris/LagerLanguageModelsDataschool/data/downloads/Vodacom_Group', 'document_name': 'interim_results', 'is_explicitly_incorrect': False, 'possibly_incorrect': False, 'missing_data': True, 'reason': 'No recent interim_results document was found.'}"""